<a href="https://colab.research.google.com/github/FELIPEACASTRO/AIForge/blob/master/arc_agi2_passk_probe_p144.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title ARC-AGI-2 · probe de pass@k (p144, celula unica) { display-mode: "form" }
#@markdown ### Runtime: **L4** (paridade EXATA com o Kaggle L4x4 — mesma arquitetura Ada sm_89, mesmo bf16). A100 vale SO para o gap de selecao (timing nao transfere). **T4 e BLOQUEADA** (sem tensor cores bf16 → o pool muda e o pass@k nao transfere).
#@markdown Credenciais vem dos **Secrets do Colab** (icone da chave 🔑) — nada e digitado aqui.
HF_REPO         = "felipesp1983/arc-agi2-passk-p143"  #@param {type:"string"}
N_TAREFAS       = 2   #@param {type:"integer"}
SEG_POR_TAREFA  = 900 #@param {type:"integer"}
SHA_ESPERADO    = ""  #@param {type:"string"}
REPROCESSAR     = False #@param {type:"boolean"}
#@markdown *REPROCESSAR=True re-mede saidas que ja estao no Drive (ignora o cache). Deixe False para retomar sessao caida.*
PERMITIR_TF32   = False #@param {type:"boolean"}
PERMITIR_GPU_SEM_BF16 = False #@param {type:"boolean"}
#@markdown *So ligue PERMITIR_GPU_SEM_BF16 para um smoke exploratorio: o resultado NUNCA vale como medida.*
PIN_TRANSFORMERS = "4.55.4"  #@param {type:"string"}
PIN_UNSLOTH      = "2025.9.7" #@param {type:"string"}
PIN_UNSLOTH_ZOO  = "2025.9.9" #@param {type:"string"}
#@markdown *unsloth 2025.9.7 exige unsloth_zoo>=2025.9.9 SEM teto — sem pinar o zoo, o pip instala um zoo 2026.x que exige transformers 5.x e DESFAZ o downgrade (medido em 2026-07-30: transformers voltou a 5.13.1 e o import morreu em BACKENDS_MAPPING/tensorflow_text). zoo 2025.9.9 aceita transformers<=4.55.4.*
#@markdown *Pin das versoes do controle (evita `training_loss=nan` por incompat.). Vazio = usar as do Colab.*
KERNEL_CONTROLE = "felipe1983/arc-agi2-3014-queue-guard-v1"  #@param {type:"string"}
URL_ARC_STRUCTURAL = "https://gist.githubusercontent.com/FELIPEACASTRO/23951bc550c47dcc8e0c4f6f9b086df1/raw/arc_structural.py"  #@param {type:"string"}
#@markdown *Comece com N_TAREFAS=2 para validar o gancho. TF32 acelera fp32 mas reduz mantissa — deixe desligado.*

import os, sys, json, time, gc, hashlib, pathlib, threading, queue as _q, traceback, subprocess, shutil
# desliga o backend TensorFlow do transformers ANTES de qualquer import dele:
# com transformers 4.55.x pinado, o TF injeta o backend 'tensorflow_text' que
# nao esta no BACKENDS_MAPPING dessa versao -> ValueError na importacao.
os.environ["USE_TF"] = "0"; os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

# ---------------------------------------------------------------- 0. secrets
# Secrets do Colab: KAGGLE_USERNAME, KAGGLE_KEY, HF_TOKEN (ou HF_KEY).
# Vantagem sobre @param: nao aparecem na celula, nao vao para o gist, e
# persistem entre sessoes — zero digitacao por run.
from google.colab import userdata
def segredo(*nomes, obrig=True):
    for n in nomes:
        try:
            v = userdata.get(n)
            if v: return v
        except Exception:
            continue
    if obrig:
        raise RuntimeError(
            f"secret ausente: {' ou '.join(nomes)}. Abra o icone da chave 🔑 na barra "
            f"lateral, crie o secret e ligue 'Acesso ao notebook'.")
    return None

KAGGLE_USERNAME = segredo('KAGGLE_USERNAME')
KAGGLE_KEY      = segredo('KAGGLE_KEY')
HF_TOKEN        = segredo('HF_TOKEN', 'HF_KEY', obrig=False)   # aceita os dois nomes
print(f"secrets ok · kaggle={KAGGLE_USERNAME} · hf={'sim' if HF_TOKEN else 'NAO (log so no Drive)'}")
# BUG DE ORDEM CORRIGIDO: o pacote `kaggle` autentica NA IMPORTACAO. Se as env
# vars nao estiverem definidas antes, qualquer `import kaggle` falha. Antes eu
# so definia isso na secao 4 — depois da secao 2, que ja importava o modulo.
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY']      = KAGGLE_KEY
T0 = time.time()
def marco(m): print(f"[{time.time()-T0:7.1f}s] {m}", flush=True)

# ---------------------------------------------------------------- 1. ambiente
import torch
assert torch.cuda.is_available(), "sem GPU — Ambiente de execucao > Alterar tipo"
_p = torch.cuda.get_device_properties(0)
# BUG MEDIDO (2026-07-30, run T4): torch.cuda.is_bf16_supported() retorna True
# em Tesla T4 (sm_75) — bf16 EMULADO, sem tensor cores: numerica diferente do
# L4x4 e 3-5x mais lento. NUNCA classificar pela flag do torch; usar compute
# capability: sm_89 (Ada/L4) = paridade exata; sm_80+ (A100/H100) = bf16 nativo
# mas arch diferente; abaixo de sm_80 = invalido para o probe.
_cap = torch.cuda.get_device_capability(0)
marco(f"GPU {_p.name} · {_p.total_memory/1024**3:.1f} GiB · sm_{_cap[0]}{_cap[1]} · torch {torch.__version__}")
if _cap == (8, 9):
    REGIME = 'L4_PARIDADE_EXATA'          # mesma arquitetura do Kaggle L4x4: pool E timing transferem
elif _cap >= (8, 0):
    REGIME = 'BF16_SO_GAP_SELECAO'        # A100/H100: bf16 nativo; gap de selecao ok; timing NAO transfere
else:
    REGIME = 'SEM_BF16_INVALIDO'          # T4/V100 (sm<80): bf16 emulado/ausente — pool nao transfere
marco(f"regime de fidelidade: {REGIME}")
if REGIME == 'SEM_BF16_INVALIDO' and not PERMITIR_GPU_SEM_BF16:
    raise RuntimeError(
        "GPU sem bf16 (ex.: T4): o pool NAO transfere para o Kaggle L4x4. "
        "Troque o runtime (Ambiente de execucao > Alterar tipo > L4/A100) ou, "
        "APENAS para smoke exploratorio, ligue PERMITIR_GPU_SEM_BF16.")
# TF32 muda numerica de matmul fp32 -> default OFF
torch.backends.cuda.matmul.allow_tf32 = bool(PERMITIR_TF32)
torch.backends.cudnn.allow_tf32       = bool(PERMITIR_TF32)
torch.backends.cudnn.benchmark        = True      # autotune de conv; nao afeta numerica de matmul
try:
    torch.backends.cuda.enable_flash_sdp(True); torch.backends.cuda.enable_mem_efficient_sdp(True)
except Exception:
    pass

# ---------------------------------------------------------------- 2. deps (condicional)
# BUG CORRIGIDO: a versao anterior fazia `__import__(m)` e tratava QUALQUER
# excecao como "nao instalado". O pacote `kaggle` autentica na importacao e
# levanta erro se faltar credencial — logo era classificado como ausente,
# reinstalado, e classificado como ausente de novo -> RuntimeError eterno.
# `find_spec` responde "esta instalado?" SEM executar o modulo.
import importlib.util as _ilu
def falta(m):
    try:
        return _ilu.find_spec(m) is None
    except (ImportError, ValueError, ModuleNotFoundError):
        return True
# PIN DE VERSOES — a causa raiz do `training_loss=nan`: o codigo do controle foi
# escrito p/ unsloth 2025.9.7 / transformers 4.55.4; as versoes novas do Colab
# tratam tie_word_embeddings + LoRA em embed_tokens/lm_head de forma diferente e
# o TTT diverge p/ NaN. Pinamos as versoes do controle ANTES de importar unsloth.
# BUG MEDIDO (2026-07-30): a versao antiga checava com __import__, o que CARREGAVA
# o transformers 5.13.1 na memoria ANTES do downgrade — mesmo com pip ok, a sessao
# ficava envenenada. Agora: (1) checa por importlib.metadata SEM importar;
# (2) pina tambem o unsloth_zoo; (3) apos downgrade bem-sucedido REINICIA o
# runtime sozinho (pip persiste; na re-execucao os pins ja batem e segue limpo).
import importlib.metadata as _im
def _ver(pkg):
    try: return _im.version(pkg)
    except Exception: return None
_ALVOS = {}
if PIN_TRANSFORMERS: _ALVOS['transformers'] = PIN_TRANSFORMERS
if PIN_UNSLOTH:      _ALVOS['unsloth'] = PIN_UNSLOTH
if PIN_UNSLOTH_ZOO:  _ALVOS['unsloth_zoo'] = PIN_UNSLOTH_ZOO
_errados = {p: (_ver(p), alvo) for p, alvo in _ALVOS.items() if _ver(p) != alvo}
if os.environ.get('P144_SANDBOX') == '1':
    if _errados: marco(f"[sandbox] pin pulado (ambiente de teste): {_errados}")
    _errados = {}
if _errados:
    _pins = [f"{p}=={alvo}" for p, alvo in _ALVOS.items()]
    marco(f"pinando versoes do controle: {_pins} (downgrade + reinstalar, ~3-5 min)")
    _r = subprocess.run([sys.executable,"-m","pip","install","-q",*_pins],
                        capture_output=True, text=True)
    if _r.returncode != 0:
        print("*** pip do PIN retornou erro:\n", (_r.stderr or "")[-1500:], flush=True)
    _ainda = {p: (_ver(p), alvo) for p, alvo in _ALVOS.items() if _ver(p) != alvo}
    if _ainda:
        raise RuntimeError(
            f"PIN nao aplicou: {_ainda}. O pip resolveu contra os pins (conflito de deps). "
            "Nao adianta seguir — o import do controle vai falhar. Veja o stderr do pip acima.")
    # SEM reinicio: nada importa transformers/unsloth/peft ANTES deste bloco
    # (a checagem usa importlib.metadata, que le o disco sem importar), entao a
    # sessao esta limpa e podemos seguir DIRETO na mesma execucao. O reinicio
    # antigo era herança do checador via __import__ (que envenenava a memoria) e
    # confundia o operador: cada troca de GPU = maquina nova = pin de novo =
    # "reiniciando" de novo, parecendo um erro repetido. Guarda de seguranca:
    for _mod in ('transformers', 'unsloth', 'peft', 'trl'):
        assert _mod not in sys.modules, (
            f"'{_mod}' ja importado antes do pin — ordem da celula quebrada; "
            "NAO siga: reinicie o runtime e reporte este bug.")
    marco("PIN aplicado e verificado — seguindo DIRETO (sem reinicio)")
# BUG MEDIDO (2026-07-30, run L4): peft novo (0.19.x) LEVANTA ImportError no
# dispatcher LoRA se houver torchao < 0.16 instalado ("Found version 0.10.0") —
# e o Colab traz torchao 0.10.0 de fabrica. O controle nao usa torchao (sem
# quantizacao). Remover o orfao ANTES de qualquer import de unsloth/peft faz
# is_torchao_available() responder False limpo e o dispatcher pular o caminho.
if os.environ.get('P144_SANDBOX') != '1':
    _tao = _ver('torchao')
    if _tao and tuple(int(x) for x in _tao.split('.')[:2]) < (0, 16):
        marco(f"removendo torchao {_tao} orfao (peft novo exige >=0.16; o controle nao usa)")
        subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","torchao"], check=False)
        assert _ver('torchao') is None, "torchao ainda instalado apos uninstall"

_DEPS = (("kagglehub","kagglehub"), ("huggingface_hub","huggingface_hub"),
         ("kaggle","kaggle"), ("unsloth","unsloth"), ("trl","trl"), ("peft","peft"),
         ("datasets","datasets"))   # o worker faz `from datasets import Dataset`
_pend = [p for m, p in _DEPS if falta(m)]
if _pend:
    marco(f"instalando faltantes: {_pend}")
    subprocess.run([sys.executable,"-m","pip","install","-q",*_pend], check=False)
    for m,_ in _DEPS:
        if falta(m): raise RuntimeError(f"dependencia '{m}' nao instalou — veja o log do pip acima")
marco("deps ok")
try:
    import unsloth, transformers, peft, trl
    _vt, _vu = transformers.__version__, unsloth.__version__
    marco(f"versoes · unsloth={_vu} transformers={_vt} peft={peft.__version__} trl={trl.__version__}")
    if PIN_TRANSFORMERS and _vt != PIN_TRANSFORMERS:
        print(f"\n*** AVISO: transformers={_vt} != pin {PIN_TRANSFORMERS}. O pin nao aplicou\n"
              "*** (reinicie o runtime: Ambiente de execucao > Reiniciar sessao, e rode de novo).\n"
              "*** Com a versao errada o treino pode dar training_loss=nan e o pool sair vazio.\n", flush=True)
except Exception as _e:
    marco(f"nao consegui ler versoes: {_e}")

# ---------------------------------------------------------------- 3. Drive
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
RAIZ = pathlib.Path('/content/drive/My Drive/arc_agi2_passk_p143')
DUMP = RAIZ/'pool'; LOG = RAIZ/'log'
for d in (RAIZ, DUMP, LOG): d.mkdir(parents=True, exist_ok=True)
marco(f"Drive ok · {len(list(DUMP.glob('*.json')))} saidas ja feitas")

# ---------------------------------------------------------------- 4. modelo e dados (SEM copia dupla)
import kagglehub   # env vars ja definidas na secao 0
LOCAL = pathlib.Path('/content/arc'); LOCAL.mkdir(exist_ok=True)
_c = kagglehub.competition_download('arc-prize-2026-arc-agi-2')
for n in ('arc-agi_evaluation_challenges.json','arc-agi_evaluation_solutions.json'):
    if not (LOCAL/n).exists(): shutil.copy(os.path.join(_c,n), LOCAL/n)
# o modelo fica ONDE ESTA (cache do kagglehub, disco local do Colab). Copiar 6,77 GB
# de novo custaria minutos e nao acelera nada — o cache ja e /root/.cache, nao Drive.
MODELO = kagglehub.model_download('sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1')
# VALIDACAO DURA do que o kagglehub devolveu (pode dar caminho sem baixar pesos).
# NAO fazemos symlink: em vez disso, o patch da secao 5 insere ESTE caminho como
# candidato do resolve_qwen_model_dir do controle — robusto a /kaggle read-only.
_mp  = pathlib.Path(MODELO)
_cfg = _mp/'config.json'
_stf = sorted(_mp.glob('*.safetensors'))
_gb  = sum(p.stat().st_size for p in _stf)/1024**3
assert _cfg.is_file(), (
    f"config.json ausente em {MODELO} — o modelo NAO foi baixado. "
    f"Cheque a credencial/quota do Kaggle e rode de novo.")
assert _stf and _gb > 6.0, f"pesos incompletos: {len(_stf)} shards, {_gb:.2f} GiB (esperado ~6.77)"
marco(f"modelo em {MODELO} · {len(_stf)} shards · {_gb:.2f} GiB")

ch  = json.load(open(LOCAL/'arc-agi_evaluation_challenges.json'))
sol = json.load(open(LOCAL/'arc-agi_evaluation_solutions.json'))

# ---------------------------------------------------------------- 5. travas
CONTAMINADAS = ['0934a4d8', '136b0064', '16b78196', '981571dc', 'aa4ec2a5', 'da515329']
vaz = [(t,i) for t,v in ch.items() for i,p in enumerate(v['test']) if 'output' in p]
assert not vaz, f"VAZAMENTO: {len(vaz)} pares de test com 'output'"
LIMPAS = [t for t in sorted(ch) if t not in CONTAMINADAS]
assert len(LIMPAS) == len(ch)-len(CONTAMINADAS)
marco(f"travas ok · {len(LIMPAS)} tarefas limpas ({len(CONTAMINADAS)} contaminadas fora)")

# --- controle: busca automatica, sem upload manual ---------------------
# Prioridade: (1) Drive, se ja existir; (2) pull direto do kernel do Kaggle.
# O (2) e melhor: garante que e o fonte QUE PONTUOU 30.56, nao uma copia velha.
CTRL = pathlib.Path('/content/controle'); CTRL.mkdir(exist_ok=True)
BUNDLE = RAIZ/'controle'
if BUNDLE.exists() and any(BUNDLE.iterdir()):
    for p in BUNDLE.rglob('*'):
        if p.is_file(): shutil.copy(p, CTRL/p.name)
    marco(f"controle do Drive ({len(list(CTRL.iterdir()))} arquivos)")
else:
    marco("controle nao esta no Drive — puxando o kernel do Kaggle")
    subprocess.run([sys.executable,"-m","pip","install","-q","kaggle"], check=False)
    r = subprocess.run(["kaggle","kernels","pull",KERNEL_CONTROLE,"-p",str(CTRL),"-m"],
                       capture_output=True, text=True)
    print(r.stdout or "", r.stderr or "")
    nbs = list(CTRL.glob('*.ipynb'))
    assert nbs, f"pull falhou; suba o bundle manualmente para {BUNDLE}"
    # o notebook do controle escreve os proprios modulos (%%writefile) — extraimos
    _src = "\n".join("".join(c.get('source',[]))
                     for c in json.load(open(nbs[0]))['cells'] if c['cell_type']=='code')
    (CTRL/'controle_fonte.py').write_text(_src, encoding='utf-8')
    import re as _re
    for m in _re.finditer(r"%%writefile\s+(\S+\.py)\n(.*?)(?=\n%%writefile|\Z)", _src, _re.S):
        (CTRL/os.path.basename(m.group(1))).write_text(m.group(2), encoding='utf-8')
    marco(f"controle extraido: {sorted(p.name for p in CTRL.glob('*.py'))}")
    BUNDLE.mkdir(parents=True, exist_ok=True)          # cacheia no Drive p/ proxima sessao
    for p in CTRL.iterdir():
        if p.is_file(): shutil.copy(p, BUNDLE/p.name)

_arqs = sorted((p for p in CTRL.rglob('*') if p.is_file()), key=lambda x: x.name)
_h = hashlib.sha256()
for p in _arqs: _h.update(p.name.encode()); _h.update(p.read_bytes())
SHA = _h.hexdigest()
if SHA_ESPERADO:
    assert SHA == SHA_ESPERADO, f"INTEGRIDADE FALHOU: {SHA}"
    marco(f"bundle integro ({len(_arqs)} arquivos)")
else:
    marco(f"bundle sha256={SHA}  <- anote e preencha SHA_ESPERADO no run definitivo")

# PATCH DE CAMINHOS (nao de logica): o worker le o eval, escreve o pool e
# resolve o modelo em caminhos /kaggle/... read-only. Redirecionamos para
# caminhos gravaveis com os MESMOS bytes — o worker processa o mesmo dado, so
# muda de onde le/escreve. IDEMPOTENTE: detecta se ja foi patchado (cache do
# Drive pode conter versao patchada) e nao re-aplica.
_ap = CTRL/'arc_solver.py'
_asrc = _ap.read_text(encoding='utf-8')
_MARCA = "# __PATCHED_FOR_COLAB__"
if _MARCA in _asrc:
    marco("arc_solver.py ja patchado (idempotente) — pulando")
else:
    _EVAL_LOCAL = str(LOCAL/'arc-agi_evaluation_challenges.json')
    _n1 = _asrc.count('arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json')
    _n2 = _asrc.count('"/kaggle/inference_outputs"')
    _n3 = _asrc.count('candidates = (')
    assert _n1 >= 1 and _n2 >= 1 and _n3 >= 1, (
        f"patch nao achou os pontos esperados no arc_solver (eval={_n1}, out={_n2}, "
        f"resolve={_n3}). O fonte do controle mudou — reveja antes de rodar.")
    _asrc = _asrc.replace(
        '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json', _EVAL_LOCAL)
    _asrc = _asrc.replace(
        '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json', _EVAL_LOCAL)
    _asrc = _asrc.replace('"/kaggle/inference_outputs"', '"/content/inference_outputs"')
    # insere o caminho REAL do modelo como 1o candidato do resolve (robusto a
    # read-only): nao dependemos mais de symlink em /kaggle/input.
    _asrc = _asrc.replace('candidates = (', f'candidates = (\n        {json.dumps(MODELO)},', 1)
    _asrc = _MARCA + "\n" + _asrc
    _ap.write_text(_asrc, encoding='utf-8')
    marco(f"arc_solver.py: eval+saida+modelo redirecionados p/ Colab")
sys.path.insert(0, str(CTRL))

# --- arc_structural.py: baixa do gist, sem upload manual ---------------
_st = pathlib.Path('/content/arc_structural.py')
if not _st.exists():
    if (RAIZ/'arc_structural.py').exists():
        shutil.copy(RAIZ/'arc_structural.py', _st)
    else:
        import urllib.request
        urllib.request.urlretrieve(URL_ARC_STRUCTURAL, _st)
        marco("arc_structural.py baixado do gist")
sys.path.insert(0,'/content')
from arc_structural import forma_obrigatoria

# ---------------------------------------------------------------- 6. amostra estratificada
import random
rnd = random.Random(20260730)
com, sem = [], []
for t in LIMPAS:
    for i in range(len(sol[t])):
        (com if forma_obrigatoria(ch[t], i) is not None else sem).append((t,i))
rnd.shuffle(com); rnd.shuffle(sem)
_m = N_TAREFAS//2
AMOSTRA = com[:_m] + sem[:N_TAREFAS-_m]; rnd.shuffle(AMOSTRA)
marco(f"amostra: {len(AMOSTRA)} saidas ({_m} com forma obrigatoria / {len(AMOSTRA)-_m} sem)")

# ---------------------------------------------------------------- 7. HF assincrono
hf_api = None; _fila = _q.Queue()
if HF_TOKEN:
    from huggingface_hub import HfApi
    hf_api = HfApi(token=HF_TOKEN)
    hf_api.create_repo(HF_REPO, repo_type='dataset', private=True, exist_ok=True)
    # FATAL_* de runs ANTIGOS confundem o diagnostico (parecem erro do run atual).
    try:
        for _f in hf_api.list_repo_files(HF_REPO, repo_type='dataset'):
            if _f.startswith('FATAL_'):
                hf_api.delete_file(_f, repo_id=HF_REPO, repo_type='dataset')
                print(f"[hf] removido {_f} (relíquia de run anterior)", flush=True)
    except Exception as _e:
        print("[hf] limpeza de FATAL_* falhou (segue):", str(_e)[:80], flush=True)
    def _bombeia():
        while True:
            it = _fila.get()
            if it is None: break
            nome, txt = it
            try: hf_api.upload_file(path_or_fileobj=txt.encode(), path_in_repo=nome,
                                    repo_id=HF_REPO, repo_type='dataset')
            except Exception as e: print("[hf]", str(e)[:90], flush=True)
            finally: _fila.task_done()
    threading.Thread(target=_bombeia, daemon=True).start()
    marco(f"log ao vivo: https://hf.co/datasets/{HF_REPO}")
def log_hf(nome, obj):
    if hf_api: _fila.put((nome, json.dumps(obj, ensure_ascii=False)))

# ------------------------------------------------ 7-bis. STATUS ROLANTE + GO/NO-GO
# Contrato com quem monitora pelo HF: leia `status.json` e OBEDECA `recomendacao`.
#   CONTINUAR        -> deixe o job rodar
#   ABORTAR_INVALIDO -> desligue o runtime JA (resultado nao vale; queima sessao a toa)
#   ABORTAR_ESTAGNADO-> worker sem progresso alem do orcamento; investigar/desligar
# Criterios (explicitos, avaliados a cada batida):
#   [A] REGIME sem bf16            -> ABORTAR_INVALIDO (pool nao transfere)
#   [B] training_loss=nan          -> ABORTAR_INVALIDO (stack incompativel; lixo)
#   [C] worker > 1.25x orcamento com ZERO arquivo de pool -> ABORTAR_ESTAGNADO
#   [D] >=5 saidas medidas e mediana n_cands < 4 (esperado ~16) -> ABORTAR_INVALIDO
ESTADO = {"fase":"inicio","gpu":_p.name,"regime":REGIME,"nan":False,
          "pool_files":0,"saidas_medidas":0,"n_cands_hist":[],
          "worker_t0":None,"worker_orcamento_s":None}
def _recomendacao():
    if ESTADO["regime"] == 'SEM_BF16_INVALIDO': return "ABORTAR_INVALIDO", "[A] GPU sem bf16"
    if ESTADO["nan"]: return "ABORTAR_INVALIDO", "[B] training_loss=nan no TTT"
    if ESTADO["fase"] == "worker" and ESTADO["worker_t0"] and ESTADO["worker_orcamento_s"]:
        _el = time.time() - ESTADO["worker_t0"]
        if _el > 1.25*ESTADO["worker_orcamento_s"] and ESTADO["pool_files"] == 0:
            return "ABORTAR_ESTAGNADO", f"[C] {_el/60:.0f}min sem NENHUM arquivo de pool"
    h = ESTADO["n_cands_hist"]
    if len(h) >= 5:
        _med = sorted(h)[len(h)//2]
        if _med < 4: return "ABORTAR_INVALIDO", f"[D] mediana n_cands={_med} (esperado ~16)"
    return "CONTINUAR", "sem criterio de aborto disparado"
def publica_status(fase=None):
    if fase: ESTADO["fase"] = fase
    rec, mot = _recomendacao()
    _s = {"ts":time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
          "elapsed_min":round((time.time()-T0)/60,1), "recomendacao":rec, "motivo":mot, **{
          k:ESTADO[k] for k in ("fase","gpu","regime","nan","pool_files","saidas_medidas")},
          "n_cands_mediana":(sorted(ESTADO["n_cands_hist"])[len(ESTADO["n_cands_hist"])//2]
                              if ESTADO["n_cands_hist"] else None)}
    try: (LOG/'status.json').write_text(json.dumps(_s), encoding='utf-8')
    except Exception: pass
    log_hf('status.json', _s)
    if rec != "CONTINUAR": print(f"\n*** STATUS: {rec} — {mot}\n", flush=True)
    return rec
_hb_stop = threading.Event()
def _heartbeat():
    while not _hb_stop.wait(120):                       # batida a cada 2 min
        try:
            ESTADO["pool_files"] = len(list(pathlib.Path('/content/inference_outputs').glob('*')))
        except Exception: pass
        publica_status()
publica_status("preparacao")

# ---------------------------------------------------------------- 8. gancho no controle
# Handler de erro FATAL: grava QUALQUER excecao daqui em diante no HF e no Drive,
# para o diagnostico chegar mesmo quando o output do Colab some/trunca.
def _fatal(rotulo):
    _tb = traceback.format_exc()
    print(f"\n*** ERRO FATAL em '{rotulo}':\n{_tb}", flush=True)
    try: (LOG/f"FATAL_{rotulo}.txt").write_text(_tb, encoding='utf-8')
    except Exception: pass
    try:
        log_hf(f"FATAL_{rotulo}.json", {"rotulo": rotulo, "erro": _tb[-2500:]})
        for _ in range(30):
            if _fila.empty(): break
            time.sleep(1)
    except Exception: pass

# REESCRITO apos ler o fonte inteiro do worker. Os simbolos moram em arquivos
# distintos (nao todos em arc_solver). Import sob guarda: se o PIN de versao
# quebrou o ambiente (transformers 4.55 + unsloth 2025.9.7 no Colab), a falha e
# aqui — e agora ela vai para o HF, nao some no output.
try:
    from arc_loader import ArcDataset
    from arc_decoder import ArcDecoder, score_kgmon       # scorer e leitor de pool
    from arc_solver import worker                         # o trabalho: TTT + DFS
except Exception:
    _fatal("import_controle")
    import transformers as _tf
    print("*** Provavel ambiente quebrado pelo PIN. transformers instalado:",
          getattr(_tf, "__version__", "?"),
          "\n*** Se o downgrade nao aplicou limpo, tente: PIN_TRANSFORMERS e PIN_UNSLOTH\n"
          "*** VAZIOS (usa o stack do Colab) — pode dar NaN, mas ao menos importa;\n"
          "*** ou rode o probe no proprio Kaggle, onde o stack e o nativo do controle.",
          flush=True)
    raise

# O worker do controle NAO recebe dataset: ele le de um caminho FIXO (agora
# redirecionado para LOCAL pelo patch acima) e roda uma FILA de tarefas num
# loop (uma carga de modelo, N tarefas). Escreve o pool em /content/inference_outputs
# (tambem redirecionado). Nada mais em /kaggle read-only.
os.environ.pop('KAGGLE_IS_COMPETITION_RERUN', None)   # forca modo eval (nao rerun)
STORE = pathlib.Path('/content/inference_outputs')
STORE.mkdir(parents=True, exist_ok=True)
for _p in STORE.glob('*'):
    _p.unlink()

def _grade(g):
    if hasattr(g, 'tolist'): g = g.tolist()          # np.array (campo 'solution')
    return [[int(x) for x in linha] for linha in g]

# --- roda o worker sobre as tarefas que FALTAM ---------------------------
# BUG MEDIDO (2026-07-30): o worker rodava a tarefa INTEIRA na GPU e so depois
# a secao 9 descobria que a saida ja estava no Drive -> 11,7min de L4 gastos
# para produzir zero medicao nova. Pior no run real: sessao que cai na saida 20
# de 30 refazia as 20 na GPU ao retomar (horas perdidas). Agora a fila do worker
# leva SO as tarefas com pelo menos uma saida ainda nao medida.
_TODAS = sorted({tid for tid, ti in AMOSTRA})
if REPROCESSAR:
    TASKS = _TODAS
else:
    TASKS = sorted({tid for tid, ti in AMOSTRA if not (DUMP/f"{tid}__{ti}.json").exists()})
    _ja = len(_TODAS) - len(TASKS)
    if _ja:
        marco(f"cache: {_ja} de {len(_TODAS)} tarefas ja medidas — worker roda so as {len(TASKS)} que faltam")
if not TASKS:
    print("\n*** Nada a fazer: TODAS as saidas da amostra ja estao medidas no Drive.\n"
          "*** Aumente N_TAREFAS (as novas entram sem refazer as antigas) ou marque\n"
          "*** REPROCESSAR para re-medir. NENHUMA GPU sera gasta agora.\n", flush=True)
q = _q.Queue()
for tid in TASKS: q.put(tid)
q.put(None)                                            # sentinela: 1 por worker
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
marco(f"worker: {len(TASKS)} tarefas na fila, orcamento {len(TASKS)*SEG_POR_TAREFA/60:.0f}min")
_tw = time.time()
ESTADO["worker_t0"] = _tw; ESTADO["worker_orcamento_s"] = len(TASKS)*SEG_POR_TAREFA
publica_status("worker")
threading.Thread(target=_heartbeat, daemon=True).start()   # status.json a cada 2 min durante o worker
# ⚠️ ESTE e o ponto de risco de RUNTIME que a analise estatica nao cobre: o
# codigo do controle foi escrito para unsloth 2025.9.7 / transformers 4.55.4;
# o Colab pode ter versoes mais novas. Se o worker crashar por API incompativel,
# damos um diagnostico claro (versoes + dica) em vez de um traceback cru.
# tee do stdout do worker: mostra ao vivo E guarda p/ detectar NaN no treino.
import io as _io
from contextlib import redirect_stdout as _rso
class _Tee:
    def __init__(self, real): self.real, self.buf = real, _io.StringIO()
    def write(self, s): self.real.write(s); self.buf.write(s); return len(s)
    def flush(self): self.real.flush()
_tee = _Tee(sys.stdout)
if TASKS:                       # fila vazia = tudo em cache: nao carrega modelo nem gasta GPU
    try:
        with _rso(_tee):
            worker(0, q, time.time() + len(TASKS)*SEG_POR_TAREFA)  # carrega modelo 1x, processa a fila
    except Exception:
        _fatal("worker")
        raise
else:
    marco("worker PULADO (fila vazia — tudo em cache); nenhuma GPU gasta")
_wlog = _tee.buf.getvalue()
_NAN = ('training_loss=nan' in _wlog) or ('loss=nan' in _wlog.replace(' ',''))
_hb_stop.set()                                          # encerra o heartbeat do worker
ESTADO["nan"] = bool(_NAN)
ESTADO["pool_files"] = len(list(STORE.glob('*')))
publica_status("pos_worker")
marco(f"worker terminou em {(time.time()-_tw)/60:.1f}min · "
      f"{len(list(STORE.glob('*')))} arquivos de saida")
if _NAN:
    import transformers as _tf, unsloth as _us
    print("\n" + "="*70 +
          "\n*** RESULTADO INVALIDO: training_loss=nan detectado no TTT.\n"
          f"*** O treino divergiu — stack instalado (unsloth={_us.__version__}, "
          f"transformers={_tf.__version__}) incompativel com o controle.\n"
          "*** O modelo produziu lixo; os pass@k abaixo NAO sao reais.\n"
          "*** ACAO: garanta PIN_TRANSFORMERS=4.55.4 e PIN_UNSLOTH=2025.9.7, e se o\n"
          "*** pin so aplicou apos importar unsloth, REINICIE a sessao e rode de novo.\n"
          + "="*70 + "\n", flush=True)

# --- le o pool e a selecao oficial ---------------------------------------
# run_selection_algo devolve, por base_key '<task>_<idx>', a lista de solucoes
# ordenada pelo score_kgmon (o mesmo do 30.56).
_orig = ArcDataset.from_file(str(LOCAL/'arc-agi_evaluation_challenges.json'))
dec = ArcDecoder(dataset=_orig, n_guesses=2)
dec.load_decoded_results(str(STORE))
_sel = dec.run_selection_algo(score_kgmon)             # {base_key: [solucao_ordenada, ...]}
marco(f"decoded_results: {len(dec.decoded_results)} base_keys · "
      f"selecao: {len(_sel)} base_keys")
# GUARDA de plumbing: decoded_results TOTALMENTE vazio com o worker tendo rodado
# significa que o pool nao foi lido — caminho errado, worker abortou no load do
# modelo, etc. Distinto de UMA tarefa sem candidatos (que e legitimo). Avisa
# alto, mas nao aborta: os dumps por-saida (n_cands=0) ainda registram o estado.
if not dec.decoded_results:
    print("\n*** ATENCAO: decoded_results VAZIO apos o worker. O pool nao foi\n"
          "*** extraido — cheque se o worker carregou o modelo e escreveu em\n"
          f"*** {STORE} (arquivos: {[p.name for p in STORE.glob('*')][:5]}).\n"
          "*** Os numeros abaixo serao todos pass@k=False; NAO confie neles.\n", flush=True)

# ---------------------------------------------------------------- 9. metricas por saida
# BUG MEDIDO (2026-07-30): saidas ja no Drive eram puladas em SILENCIO e ficavam
# FORA das estatisticas -> status.json reportava "saidas=0" mesmo com o worker
# tendo rodado (a amostra e deterministica: re-rodar com o mesmo N repete as
# mesmas saidas). Agora: contabiliza as puladas nas metricas, diz quantas foram,
# e REPROCESSAR=True ignora o cache.
feitos = 0; pulados = 0
for (tid, ti) in AMOSTRA:
    chave = f"{tid}__{ti}"; alvo = DUMP/f"{chave}.json"
    if alvo.exists() and not REPROCESSAR:
        pulados += 1
        try:                                  # recupera a estatistica do cache
            _rec = json.loads(alvo.read_text(encoding='utf-8'))
            _nc = len(_rec.get('candidates', []))
            ESTADO["n_cands_hist"].append(_nc)
            print(f"[cache] {chave} ja medida ({_nc} cands) — pulando "
                  f"(REPROCESSAR=True para refazer)", flush=True)
        except Exception:
            print(f"[cache] {chave} ja existe mas nao pude ler — pulando", flush=True)
        continue
    try:
        bk = f"{tid}_{ti}"                             # base_key do controle (fill_submission)
        bruto = dec.decoded_results.get(bk, {})
        # ORDENADO pela selecao oficial (mesmo scorer do 30.56). _sel[bk] ja vem
        # deduplicado e ranqueado — e a ordem que importa para pass@2 e rank.
        ordenado = [_grade(s) for s in _sel.get(bk, [])]
        norm = [json.dumps(c, separators=(',',':')) for c in ordenado]
        gt = json.dumps(sol[tid][ti], separators=(',',':'))
        pk = gt in norm                               # certo em ALGUM candidato
        p2 = gt in norm[:2]                            # certo entre os 2 enviados
        # BUG CORRIGIDO: rank_do_certo agora e a posicao na lista ORDENADA por
        # score (nao na ordem de insercao do dict). E o numero que decide se o
        # certo estava no pool mas foi mal-ranqueado.
        rk = norm.index(gt) if pk else -1
        alvo.write_text(json.dumps({"task_id":tid,"test_index":ti,"candidates":ordenado,
                                    "selected":[0,1][:min(2,len(ordenado))],
                                    "rank_do_certo":rk,"base_key_presente":bool(bruto)}),
                        encoding='utf-8')
        feitos += 1
        print(f"[{feitos}/{len(AMOSTRA)}] {chave} · {len(ordenado)} cands · "
              f"pass@k={pk} pass@2={p2} rank_do_certo={rk}", flush=True)
        log_hf(f"live/{chave}.json", {"chave":chave,"n_cands":len(ordenado),
               "pass_k":pk,"pass_2":p2,"rank_do_certo":rk,
               "base_key_presente":bool(bruto),
               "forma_obrigatoria":str(forma_obrigatoria(ch[tid],ti)),
               "vram_pico_gib":round(torch.cuda.max_memory_allocated()/1024**3,2)})
        ESTADO["saidas_medidas"] = feitos
        ESTADO["n_cands_hist"].append(len(ordenado))
        publica_status("medicao")
    except Exception:
        tb = traceback.format_exc()
        print(f"[ERRO] {chave}\n{tb[-800:]}", flush=True)
        try:
            LOG.mkdir(parents=True, exist_ok=True)
            (LOG/f"erro_{chave}.txt").write_text(tb, encoding='utf-8')
        except Exception as _le:
            print(f"[ERRO] nao gravou log em {LOG}: {_le}", flush=True)
        try: log_hf(f"erros/{chave}.json", {"chave":chave,"erro":tb[-600:]})
        except Exception: pass
        print(f"[ERRO] {chave} — registrado, seguindo", flush=True)

# ---------------------------------------------------------------- 10. consolidacao
saida = RAIZ/'passk_dump.jsonl'
with open(saida,'w',encoding='utf-8') as f:
    n = 0
    for p in sorted(DUMP.glob('*.json')):
        f.write(p.read_text(encoding='utf-8').strip()+"\n"); n += 1
if hf_api:
    log_hf('passk_dump.jsonl', {"n": n})
    for _ in range(60):
        if _fila.empty(): break
        time.sleep(1)
ESTADO["saidas_medidas"] = feitos + pulados      # o que existe de fato no dump
publica_status("concluido")
marco(f"FIM · {feitos} medidas agora + {pulados} do cache = {feitos+pulados} · "
      f"{n} no dump · total {(time.time()-T0)/60:.1f}min")
if pulados and not feitos:
    print(f"\n*** NOTA: as {pulados} saidas da amostra JA estavam no Drive — nada novo foi\n"
          f"*** medido. A amostra e deterministica: para medir MAIS, aumente N_TAREFAS\n"
          f"*** (as novas entram sem refazer as antigas) ou marque REPROCESSAR.\n", flush=True)
print(f"\ndump: {saida}")
print("analise LOCAL (sem GPU):  python passk_analyzer.py passk_dump.jsonl")


secrets ok · kaggle=felipe1983 · hf=sim
[    4.4s] GPU NVIDIA L4 · 22.0 GiB · sm_89 · torch 2.11.0+cu128
[    4.4s] regime de fidelidade: L4_PARIDADE_EXATA
[    4.4s] pinando versoes do controle: ['transformers==4.55.4', 'unsloth==2025.9.7', 'unsloth_zoo==2025.9.9'] (downgrade + reinstalar, ~3-5 min)
[   24.5s] PIN aplicado e verificado — seguindo DIRETO (sem reinicio)
[   24.5s] removendo torchao 0.10.0 orfao (peft novo exige >=0.16; o controle nao usa)
[   25.6s] deps ok
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


[   51.8s] versoes · unsloth=2025.9.7 transformers=4.55.4 peft=0.19.1 trl=0.22.2
Mounted at /content/drive
[   82.5s] Drive ok · 2 saidas ja feitas


100%|██████████| 487k/487k [00:00<00:00, 616kB/s]

Extracting files...


...


100%|██████████| 94.0/94.0 [00:00<00:00, 25.7kB/s]


100%|██████████| 68.0/68.0 [00:00<00:00, 241kB/s]




100%|██████████| 1.50k/1.50k [00:00<00:00, 1.61MB/s]




100%|██████████| 988/988 [00:00<00:00, 1.04MB/s]



  0%|          | 1.00M/2.11G [00:01<49:36, 762kB/s]
  0%|          | 1.00M/4.65G [00:01<1:57:21, 709kB/s]


  0%|          | 2.00M/2.11G [00:01<25:26, 1.49MB/s]
  0%|          | 2.00M/4.65G [00:01<1:00:06, 1.38MB/s]


  0%|          | 3.00M/2.11G [00:01<16:01, 2.36MB/s]
  0%|          | 3.00M/4.65G [00:01<38:27, 2.16MB/s]  


  0%|          | 5.00M/2.11G [00:01<08:42, 4.34MB/s]
  0%|          | 4.00M/4.65G [00:01<26:31, 3.14MB/s]


  0%|          | 7.00M/2.11G [00:02<05:42, 6.61MB/s]
  0%|          | 6.00M/4.65G [00:02<15:47, 5.27MB/s]


  0%|          | 9.00M/2.11G [00:02<04:25, 8.52MB/s]
  0%|          | 8.00M/4.65G [00:02<10:45, 7.73MB/s]


  1%|          | 11.0M/2.11G [00:02<03:30, 10.7MB/s]
  0%|          | 10.0M/4.65G [00:02<08:54, 9.33MB/s]


  1%|          | 13.0M/2.11G [00:02<03:11, 11.8MB/s]
  0%|          | 12.0M/4.65G [00:02<07:22, 11.3MB/s]


  1%|          | 15.0M/2.11G [00:02<02:47, 13.4MB/s]


  1%|          | 17.0M/

[  414.1s] modelo em /root/.cache/kagglehub/models/sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1 · 2 shards · 6.77 GiB
[  414.1s] travas ok · 114 tarefas limpas (6 contaminadas fora)


[  419.1s] controle do Drive (7 arquivos)
[  419.1s] bundle sha256=f5c5f4775fe567933205ac48ee8e805cfe6e23a07970cc8c77b3be26c48533c0  <- anote e preencha SHA_ESPERADO no run definitivo
[  419.1s] arc_solver.py: eval+saida+modelo redirecionados p/ Colab
[  419.3s] arc_structural.py baixado do gist
[  419.4s] amostra: 2 saidas (1 com forma obrigatoria / 1 sem)
[  419.9s] log ao vivo: https://hf.co/datasets/felipesp1983/arc-agi2-passk-p143
[  420.6s] cache: 2 de 2 tarefas ja medidas — worker roda so as 0 que faltam

*** Nada a fazer: TODAS as saidas da amostra ja estao medidas no Drive.
*** Aumente N_TAREFAS (as novas entram sem refazer as antigas) ou marque
*** REPROCESSAR para re-medir. NENHUMA GPU sera gasta agora.

[  420.6s] worker: 0 tarefas na fila, orcamento 0min
[  420.6s] worker PULADO (fila vazia — tudo em cache); nenhuma GPU gasta
[  420.6s] worker terminou em 0.0min · 0 arquivos de saida
[  420.6s] decoded_results: 0 base_keys · selecao: 0 base_keys

*** ATENCAO: decoded_resul